In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df_header = pd.read_csv('../data/raw/accepted_2007_to_2018Q4.csv',nrows=0)

In [4]:
print(f"Total columns: {len(df_header.columns)}")

Total columns: 151


In [5]:
print(df_header.columns.tolist())

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [6]:
loan_status_counts = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", usecols=['loan_status'])

In [7]:
print(loan_status_counts['loan_status'].value_counts())

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [8]:
target_statuses = ['Fully Paid', 'Charged Off', 'Default']

df_status = pd.read_csv(
    '../data/raw/accepted_2007_to_2018Q4.csv',
    usecols=['loan_status']
)

mask = df_status['loan_status'].isin(target_statuses)
print(f"Rows kept: {mask.sum()}")
print(f"Rows dropped: {(~mask).sum()}")

Rows kept: 1345350
Rows dropped: 915351


In [9]:
safe_features = [
    'loan_amnt', 'term', 'installment', 'emp_length', 'home_ownership',
    'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti',
    'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths',
    'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'initial_list_status',
    'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal',
    'mort_acc', 'pub_rec_bankruptcies', 'tax_liens', 'mo_sin_old_rev_tl_op',
    'num_actv_bc_tl', 'num_actv_rev_tl', 'num_tl_op_past_12m',
    'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'total_bc_limit', 'total_il_high_credit_limit'
]

cols_to_load = ['loan_status'] + safe_features

In [10]:
df = pd.read_csv('../data/raw/accepted_2007_to_2018Q4.csv', usecols = cols_to_load)

In [11]:
df = df[df['loan_status'].isin(target_statuses)].copy()

In [12]:
df['default'] = df['loan_status'].apply(lambda x: 0 if x == 'Fully Paid' else 1)

In [13]:
print(df.shape)
print(df['default'].value_counts(normalize = True))

(1345350, 42)
default
0    0.80035
1    0.19965
Name: proportion, dtype: float64


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 1345350 entries, 0 to 2260697
Data columns (total 42 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   loan_amnt                   1345350 non-null  float64
 1   term                        1345350 non-null  str    
 2   installment                 1345350 non-null  float64
 3   emp_length                  1266834 non-null  str    
 4   home_ownership              1345350 non-null  str    
 5   annual_inc                  1345350 non-null  float64
 6   verification_status         1345350 non-null  str    
 7   loan_status                 1345350 non-null  str    
 8   purpose                     1345350 non-null  str    
 9   addr_state                  1345350 non-null  str    
 10  dti                         1344976 non-null  float64
 11  earliest_cr_line            1345350 non-null  str    
 12  fico_range_low              1345350 non-null  float64
 13  fico_range_hi

In [5]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing/len(df)*100).round(2)

In [6]:
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count']>0]
print(missing_summary)

                            missing_count  missing_pct
mths_since_last_record            1116786        83.01
mths_since_last_delinq             678761        50.45
emp_length                          78516         5.84
pct_tl_nvr_dlq                      67681         5.03
avg_cur_bal                         67549         5.02
mo_sin_old_rev_tl_op                67528         5.02
num_actv_rev_tl                     67527         5.02
tot_coll_amt                        67527         5.02
num_actv_bc_tl                      67527         5.02
tot_cur_bal                         67527         5.02
total_il_high_credit_limit          67527         5.02
num_tl_op_past_12m                  67527         5.02
bc_util                             61914         4.60
percent_bc_gt_75                    61557         4.58
bc_open_to_buy                      61145         4.54
mort_acc                            47281         3.51
total_bc_limit                      47281         3.51
revol_util

In [17]:
df.to_csv('../data/processed/loan_data_filtered.csv', index=False)
print(f"Saved {df.shape[0]} rows, {df.shape[1]} columns to data/processed/loan_data_filtered.csv")

Saved 1345350 rows, 42 columns to data/processed/loan_data_filtered.csv


In [3]:
df = pd.read_csv('../data/processed/loan_data_filtered.csv')
print(df.shape)

(1345350, 42)


In [7]:
cols_check = ['avg_cur_bal', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'tot_coll_amt']
null_mask = df[cols_check].isnull().all(axis=1)
print(f"Rows missing ALL of these together: {null_mask.sum()}")

Rows missing ALL of these together: 67527


In [8]:
cluster_cols = [
    'avg_cur_bal', 'mo_sin_old_rev_tl_op', 'num_actv_rev_tl', 'tot_coll_amt',
    'num_actv_bc_tl', 'tot_cur_bal', 'total_il_high_credit_limit', 'num_tl_op_past_12m'
]

for col in cluster_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled with median = {median_val}")

avg_cur_bal: filled with median = 7407.0
mo_sin_old_rev_tl_op: filled with median = 164.0
num_actv_rev_tl: filled with median = 5.0
tot_coll_amt: filled with median = 0.0
num_actv_bc_tl: filled with median = 3.0
tot_cur_bal: filled with median = 80231.0
total_il_high_credit_limit: filled with median = 31681.0
num_tl_op_past_12m: filled with median = 2.0


In [9]:
df['never_delinq'] = df['mths_since_last_delinq'].isnull().astype(int)
df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(999)

df['never_public_record'] = df['mths_since_last_record'].isnull().astype(int)
df['mths_since_last_record'] = df['mths_since_last_record'].fillna(999)

print(df[['never_delinq', 'mths_since_last_delinq', 'never_public_record', 'mths_since_last_record']].isnull().sum())

never_delinq              0
mths_since_last_delinq    0
never_public_record       0
mths_since_last_record    0
dtype: int64


In [10]:
# Numeric columns - median fill
numeric_moderate = ['pct_tl_nvr_dlq', 'bc_util', 'percent_bc_gt_75', 'bc_open_to_buy', 'mort_acc', 'total_bc_limit']

for col in numeric_moderate:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# emp_length - categorical, fill with 'Unknown' rather than guessing a length
df['emp_length'] = df['emp_length'].fillna('Unknown')

print(df[numeric_moderate + ['emp_length']].isnull().sum())

pct_tl_nvr_dlq      0
bc_util             0
percent_bc_gt_75    0
bc_open_to_buy      0
mort_acc            0
total_bc_limit      0
emp_length          0
dtype: int64


In [11]:
negligible = ['revol_util', 'pub_rec_bankruptcies', 'dti', 'tax_liens', 'inq_last_6mths']

for col in negligible:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(df[negligible].isnull().sum())

revol_util              0
pub_rec_bankruptcies    0
dti                     0
tax_liens               0
inq_last_6mths          0
dtype: int64


In [12]:
print(df.isnull().sum().sum())

0


In [13]:
df.dtypes

loan_amnt                     float64
term                              str
installment                   float64
emp_length                        str
home_ownership                    str
annual_inc                    float64
verification_status               str
loan_status                       str
purpose                           str
addr_state                        str
dti                           float64
earliest_cr_line                  str
fico_range_low                float64
fico_range_high               float64
inq_last_6mths                float64
mths_since_last_delinq        float64
mths_since_last_record        float64
open_acc                      float64
pub_rec                       float64
revol_bal                     float64
revol_util                    float64
total_acc                     float64
initial_list_status               str
application_type                  str
acc_now_delinq                float64
tot_coll_amt                  float64
tot_cur_bal 

In [15]:
categorical_cols = df.select_dtypes(include=['object','str']).columns.tolist()
print(categorical_cols)

for col in categorical_cols:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())

['term', 'emp_length', 'home_ownership', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'application_type']

term: 2 unique values
term
36 months    1020768
60 months     324582
Name: count, dtype: int64

emp_length: 12 unique values
emp_length
10+ years    442209
2 years      121751
< 1 year     108065
3 years      107602
1 year        88495
Name: count, dtype: int64

home_ownership: 6 unique values
home_ownership
MORTGAGE    665596
RENT        534436
OWN         144840
ANY            286
OTHER          144
Name: count, dtype: int64

verification_status: 3 unique values
verification_status
Source Verified    521289
Verified           418352
Not Verified       405709
Name: count, dtype: int64

loan_status: 3 unique values
loan_status
Fully Paid     1076751
Charged Off     268559
Default             40
Name: count, dtype: int64

purpose: 14 unique values
purpose
debt_consolidation    780342
credit_card           295285
home_impr

In [16]:
df.describe()

,loan_amnt,installment,annual_inc,dti,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,acc_now_delinq,tot_coll_amt,tot_cur_bal,avg_cur_bal,bc_open_to_buy,bc_util,mo_sin_old_rev_tl_op,mort_acc,num_actv_bc_tl,num_actv_rev_tl,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,total_bc_limit,total_il_high_credit_limit,default,never_delinq,never_public_record
count,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06,1.345350e+06
mean,1.441997e+04,4.380756e+02,7.624757e+04,1.828245e+01,6.961853e+02,7.001854e+02,6.550756e-01,5.210200e+02,8.412786e+02,1.159351e+01,2.152778e-01,1.624796e+04,5.181003e+01,2.498075e+01,5.045527e-03,2.368659e+02,1.380764e+05,1.318322e+04,9.936900e+03,6.008931e+01,1.805982e+02,1.647195e+00,3.610472e+00,5.611169e+00,2.169913e+00,9.435763e+01,4.505241e+01,1.343747e-01,5.212993e-02,2.140431e+04,4.160859e+04,1.996499e-01,5.045237e-01,8.301082e-01
std,8.717098e+03,2.615145e+02,6.992485e+04,1.115887e+01,3.185280e+01,3.185345e+01,9.377688e-01,4.825711e+02,3.488077e+02,5.473848e+00,6.018645e-01,2.232791e+04,2.451329e+01,1.199851e+01,7.715997e-02,1.078368e+04,1.541173e+05,1.593085e+04,1.504131e+04,2.764293e+01,9.218234e+01,1.968853e+00,2.195600e+00,3.220937e+00,1.798573e+00,8.562440e+00,3.517697e+01,3.778424e-01,3.979050e-01,2.120666e+04,4.221466e+04,3.997373e-01,4.999797e-01,3.755379e-01
min,5.000000e+02,4.930000e+00,0.000000e+00,-1.000000e+00,6.250000e+02,6.290000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.000000e+03,2.484800e+02,4.578000e+04,1.179000e+01,6.700000e+02,6.740000e+02,0.000000e+00,3.100000e+01,9.990000e+02,8.000000e+00,0.000000e+00,5.943000e+03,3.350000e+01,1.600000e+01,0.000000e+00,0.000000e+00,3.088900e+04,3.238000e+03,1.574000e+03,3.960000e+01,1.200000e+02,0.000000e+00,2.000000e+00,3.000000e+00,1.000000e+00,9.190000e+01,1.250000e+01,0.000000e+00,0.000000e+00,8.000000e+03,1.547300e+04,0.000000e+00,0.000000e+00,1.000000e+00
50%,1.200000e+04,3.754300e+02,6.500000e+04,1.761000e+01,6.900000e+02,6.940000e+02,0.000000e+00,9.990000e+02,9.990000e+02,1.100000e+01,0.000000e+00,1.113400e+04,5.220000e+01,2.300000e+01,0.000000e+00,0.000000e+00,8.023100e+04,7.407000e+03,4.700000e+03,6.320000e+01,1.640000e+02,1.000000e+00,3.000000e+00,5.000000e+00,2.000000e+00,9.800000e+01,4.290000e+01,0.000000e+00,0.000000e+00,1.510000e+04,3.168100e+04,0.000000e+00,1.000000e+00,1.000000e+00
75%,2.000000e+04,5.807300e+02,9.000000e+04,2.405000e+01,7.100000e+02,7.140000e+02,1.000000e+00,9.990000e+02,9.990000e+02,1.400000e+01,0.000000e+00,1.975500e+04,7.070000e+01,3.200000e+01,0.000000e+00,0.000000e+00,2.028118e+05,1.793100e+04,1.178100e+04,8.390000e+01,2.260000e+02,3.000000e+00,5.000000e+00,7.000000e+00,3.000000e+00,1.000000e+02,7.500000e+01,0.000000e+00,0.000000e+00,2.740000e+04,5.479900e+04,0.000000e+00,1.000000e+00,1.000000e+00
max,4.000000e+04,1.719830e+03,1.099920e+07,9.990000e+02,8.450000e+02,8.500000e+02,8.000000e+00,9.990000e+02,9.990000e+02,9.000000e+01,8.600000e+01,2.904836e+06,8.923000e+02,1.760000e+02,1.400000e+01,9.152545e+06,8.000078e+06,9.580840e+05,5.599120e+05,3.396000e+02,8.520000e+02,5.100000e+01,3.500000e+01,6.300000e+01,3.200000e+01,1.000000e+02,1.000000e+02,1.200000e+01,8.500000e+01,1.105500e+06,2

In [17]:
print(df['dti'].describe())
print(df['dti'].sort_values(ascending=False).head(10))
print(df['dti'].sort_values(ascending=True).head(10))

count    1.345350e+06
mean     1.828245e+01
std      1.115887e+01
min     -1.000000e+00
25%      1.179000e+01
50%      1.761000e+01
75%      2.405000e+01
max      9.990000e+02
Name: dti, dtype: float64
559151     999.0
424634     999.0
440010     999.0
542190     999.0
394145     999.0
402739     999.0
1326105    999.0
411552     999.0
1259190    999.0
539536     999.0
Name: dti, dtype: float64
582690    -1.0
970216    -1.0
1065855    0.0
640009     0.0
913446     0.0
925886     0.0
902833     0.0
924835     0.0
948382     0.0
530791     0.0
Name: dti, dtype: float64


In [18]:
print(df['annual_inc'].sort_values(ascending=False).head(10))

401667    10999200.0
575104     9550000.0
971810     9522972.0
203893     9500000.0
902988     9300000.0
497500     9225000.0
35668      9000000.0
191933     8900060.0
356767     8706582.0
21835      8700000.0
Name: annual_inc, dtype: float64


In [19]:
print((df['dti'] == 999).sum())
print((df['dti'] == -1).sum())

38
2


In [20]:
df.loc[df['dti'].isin([999, -1]), 'dti'] = np.nan
median_dti = df['dti'].median()
df['dti'] = df['dti'].fillna(median_dti)
print(df['dti'].describe())

count    1.345350e+06
mean     1.825476e+01
std      9.866730e+00
min      0.000000e+00
25%      1.179000e+01
50%      1.761000e+01
75%      2.405000e+01
max      9.915700e+02
Name: dti, dtype: float64


In [21]:
print(df['annual_inc'].quantile([0.95, 0.99, 0.999]))

0.950    155000.0
0.990    250000.0
0.999    577302.0
Name: annual_inc, dtype: float64
